<a href="https://colab.research.google.com/github/Sno-7178/Cross-catalyst-estimation-of-Activation-Energy-for-reactions-of-Hydrodeoxygenation-of-propanoic-acid/blob/main/stratified_k_fold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install optuna scikit-learn xgboost lightgbm pandas optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 3.7 MB/s eta 0:00:00


In [20]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold, cross_val_score, LeaveOneGroupOut, StratifiedKFold, cross_validate

**Loading the data, combining all the features we have evaluated**

In [3]:
df = pd.read_excel('/content/dummy.xlsx').dropna()
print(f"Dataset shape : {df.shape}")
print(f"Catalysts     : {df['Catalyst'].unique()}\n")

Dataset shape : (558, 61)
Catalysts     : ['Cu' 'Ni' 'Pt' 'Rh' 'Pd']



In [4]:
df.head()

,Catalyst,Reaction,Direction,bond_type,delta_g_rxn,delta_g,pauling_electronegativity,work_function,d_band_center,d_bandwidth,...,sum_product_C-O0,sum_product_C-O1,sum_product_C=O,sum_product_O-H,sum_product_C0-C0,sum_product_C0-C1,sum_product_C0-C2,sum_product_C0-C3,sum_product_C1-C1,sum_product_C1-C2
0,Cu,CH3CH2COOH* + 3* → CH3CH2CO*** + OH*,Forward,OH_removal,0.69,1.41,1.9,4.992,-2.446,0.95,...,0,0,1,1,1,1,0,0,0,0
1,Cu,CH3CH2COOH* + 3* → CH3CHCOOH*** + H*,Forward,dehydrogenation,0.62,1.08,1.9,4.992,-2.446,0.95,...,1,0,1,1,0,2,0,0,0,0
2,Cu,CH3CH2CO*** → CH3CH2* + CO* + *,Forward,decarbonylation,0.09,1.04,1.9,4.992,-2.446,0.95,...,0,0,1,0,0,1,0,0,0,0
3,Cu,CH3CH2CO*** + * → CH3CHCO*** + H*,Forward,dehydrogenation,0.43,1.02,1.9,4.992,-2.446,0.95,...,0,0,1,0,1,1,0,0,0,0
4,Cu,CH3CHCOOH*** + * → CH3CHCO*** + OH*,Forward,OH_removal,0.51,0.95,1.9,4.992,-2.446,0.95,...,0,0,1,1,1,1,0,0,0,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558 entries, 0 to 557
Data columns (total 61 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Catalyst                   558 non-null    object 
 1   Reaction                   558 non-null    object 
 2   Direction                  558 non-null    object 
 3   bond_type                  558 non-null    object 
 4   delta_g_rxn                558 non-null    float64
 5   delta_g                    558 non-null    float64
 6   pauling_electronegativity  558 non-null    float64
 7   work_function              558 non-null    float64
 8   d_band_center              558 non-null    float64
 9   d_bandwidth                558 non-null    float64
 10  d_band_filling             558 non-null    float64
 11  allen_en                   558 non-null    float64
 12  IP                         558 non-null    float64
 13  cohesive                   558 non-null    float64

In [6]:
df.columns

Index(['Catalyst', 'Reaction', 'Direction', 'bond_type', 'delta_g_rxn',
       'delta_g', 'pauling_electronegativity', 'work_function',
       'd_band_center', 'd_bandwidth', 'd_band_filling', 'allen_en', 'IP',
       'cohesive', 'lat_cons', 'total_stars', 't_n_c', 't_n_h', 't_n_o',
       't_mol_weight_x_dg', 'sum_LogP_x_dg', 'sum_Balaban_J_x_dg',
       'sum_HallKier_Alpha_x_dg', 'sum_mw', 'sum_logp', 'sum_balaban',
       'sum_hall', 'sum_reactant_O0', 'sum_reactant_O1', 'sum_reactant_C0',
       'sum_reactant_C1', 'sum_reactant_C2', 'sum_reactant_C3',
       'sum_reactant_C-H', 'sum_reactant_C-O0', 'sum_reactant_C-O1',
       'sum_reactant_C=O', 'sum_reactant_O-H', 'sum_reactant_C0-C0',
       'sum_reactant_C0-C1', 'sum_reactant_C0-C2', 'sum_reactant_C0-C3',
       'sum_reactant_C1-C1', 'sum_reactant_C1-C2', 'sum_product_O0',
       'sum_product_O1', 'sum_product_C0', 'sum_product_C1', 'sum_product_C2',
       'sum_product_C3', 'sum_product_C-H', 'sum_product_C-O0',
       'sum_pro

**The descriptor columns used for model building**

In [7]:
catalyst_identifier = ['Catalyst']

catalyst_features = ['pauling_electronegativity', 'work_function', 'd_band_center',
                    'd_bandwidth', 'd_band_filling', 'allen_en', 'IP', 'cohesive',
                    'lat_cons']

molecular_descriptors = ['sum_mw', 'sum_logp',
                        'sum_balaban', 'sum_hall']

reaction_features = ['total_stars',
                    't_n_h', 'direction', 'bond_type',
                    'delta_g_rxn', 'sum_reactant_O0', 'sum_reactant_O1',
                    'sum_reactant_C0', 'sum_reactant_C1', 'sum_reactant_C2',
                    'sum_reactant_C3', 'sum_reactant_C-H', 'sum_reactant_C-O0',

                    'sum_product_O0', 'sum_product_O1', 'sum_product_C0', 'sum_product_C1',
                    'sum_product_C2', 'sum_product_C3', 'sum_product_C-H',
                    'sum_product_C=O', 'sum_product_O-H']

label = ['delta_g']

print(f"Number of catalyst features: {len(catalyst_features)}")
print(f"Number of molecular descriptors: {len(molecular_descriptors)}")
print(f"Number of reaction features: {len(reaction_features)}")
print(f"Total number of features: {len(catalyst_features) + len(molecular_descriptors) + len(reaction_features)}")

Number of catalyst features: 9
Number of molecular descriptors: 4
Number of reaction features: 22
Total number of features: 35


**No. of each Bond Type present in the dataset**

In [8]:
print(f"Total number of bond types: {len(df["bond_type"].unique())}")
print(f"\n═══════════════════════════════ BOND TYPES ═══════════════════════════════\n")
print(f"{'Bond Type: ':<30} {'Occurrences: ':<10}\n")
for r in df["bond_type"].unique():
  print(f"{r:<30} {len(df[df["bond_type"]==r]):<10}")

Total number of bond types: 16

═══════════════════════════════ BOND TYPES ═══════════════════════════════

Bond Type:                     Occurrences: 

OH_removal                     30        
dehydrogenation                175       
decarbonylation                25        
hydrogenation                  176       
decarboxylation                14        
CC_scission                    18        
carboxyl_dissociation          10        
water_dissociation             5         
H2_dissociation                1         
CC_formation                   37        
OH_addition                    30        
carboxyl_formation             22        
water_formation                5         
H2_association                 2         
carbonylation                  5         
carboxylation                  3         


**We are doing feature engineering**
* Encoding the categorical columns of **Direction** and **Bond Type**
* Keeping the other columns as it is, we later scale the data when training the models for different algorithms

In [9]:
# One-hot encoding of categorical columns
ohe     = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_enc = ohe.fit_transform(df[['Direction', 'bond_type']])
cat_cols = ohe.get_feature_names_out(['Direction', 'bond_type'])
df_cat  = pd.DataFrame(cat_enc, columns=cat_cols, index=df.index)

# Reaction feature columns
RF_COLS_CANDIDATE = ['total_stars',
                    't_n_h', 'sum_reactant_O0', 'sum_reactant_O1',
                    'sum_reactant_C0', 'sum_reactant_C1', 'sum_reactant_C2',
                     'sum_reactant_C-H','sum_reactant_C=O', 'sum_reactant_O-H'

                    'sum_product_O0', 'sum_product_O1', 'sum_product_C0', 'sum_product_C1',
                    'sum_product_C2', 'sum_product_C-H',
                    'sum_product_C=O', 'sum_product_O-H'
]
rf_cols = [c for c in RF_COLS_CANDIDATE if c in df.columns]

# Numerical columns
NUMERIC_COLS = catalyst_features + molecular_descriptors + ["delta_g_rxn"]
num_cols = [c for c in NUMERIC_COLS if c in df.columns]

In [10]:
# ASSEMBLE FINAL FEATURE MATRIX
X            = pd.concat([df_cat, df[num_cols], df[rf_cols]], axis=1).fillna(0).astype(float) # DataFrame
y            = df[label].astype(float) # Series
print(f"Feature matrix : {X.shape}")
print(f"Target shape   : {y.shape}\n")

Feature matrix : (558, 48)
Target shape   : (558, 1)



# ══════════════ **MODEL TRAINING ON ALGORITHMS** ══════════════

In [12]:
from sklearn.model_selection import train_test_split
# Bin y into quantile-based strata
N_BINS = 5
y_binned = pd.qcut(y.squeeze(), q=N_BINS, labels=False, duplicates='drop')

# Split into working (80%) and final test (20%)
X_work, X_final_test, y_work, y_final_test, yb_work, yb_final_test = train_test_split(
    X, y, y_binned,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y_binned
)
print(f"Working data shape : {X_work.shape}")
print(f"Final test shape   : {X_final_test.shape}")

Working data shape : (446, 48)
Final test shape   : (112, 48)


# ══════════════ **RANDOM FOREST** ══════════════




# **STRATIFIED ON RANDOM FOREST**

In [ ]:
groups = df["Catalyst"].values
groups

KeyError: 'catalyst'

In [23]:
# Optuna objective with Stratified K-Fold on WORKING data only
def objective_skf(trial, X, y, y_binned):
    model = RandomForestRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 100, 500),
        max_depth        = trial.suggest_int('max_depth', 3, 12),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10),
        max_features     = trial.suggest_float('max_features', 0.3, 1.0),
        random_state     = 42,
        n_jobs           = -1
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y_binned):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        scaler = StandardScaler()
        X_tr[num_cols]  = scaler.fit_transform(X_tr[num_cols])
        X_val[num_cols] = scaler.transform(X_val[num_cols])

        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))

    return np.mean(scores)

# Run Optuna on working data only
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective_skf(trial, X_work, y_work, yb_work),  # ← X_work not X
    n_trials          = 50,
    show_progress_bar = True
)
rf_best_params = study.best_params
print("Best params :", rf_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

# Cross-validation evaluation on working data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_r2, all_mae, all_rmse, all_mse, all_mape = [], [], [], [], []

print("═══════════════════════════════ RANDOM FOREST (Stratified KFold) ═════════════\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_work, yb_work), 1):  # ← X_work
    X_train = X_work.iloc[train_idx].copy()
    X_test  = X_work.iloc[test_idx].copy()
    y_train = y_work.iloc[train_idx]
    y_test  = y_work.iloc[test_idx]

    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = RandomForestRegressor(**rf_best_params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds  = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    all_r2.append(r2);    all_mae.append(mae)
    all_rmse.append(rmse); all_mse.append(mse); all_mape.append(mape)

    print(f"Fold {fold}:  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

# CV Summary
print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
for name, vals in [("R²", all_r2), ("MAE", all_mae),
                   ("RMSE", all_rmse), ("MSE", all_mse), ("MAPE%", all_mape)]:
    fmt = ".2f" if name == "MAPE%" else ".4f"
    print(f"{name:<8} {np.mean(vals):>10{fmt}} {np.std(vals):>10{fmt}} "
          f"{np.min(vals):>10{fmt}} {np.max(vals):>10{fmt}}")

# Train final model on ALL working data, evaluate on held-out test
print("\n═══════════════════════════════ FINAL TEST SET ════════════════════════════════\n")

scaler_final        = StandardScaler()
X_work_scaled       = X_work.copy()
X_final_test_scaled = X_final_test.copy()

X_work_scaled[num_cols]       = scaler_final.fit_transform(X_work[num_cols])
X_final_test_scaled[num_cols] = scaler_final.transform(X_final_test[num_cols])  # ← no fit!

final_model = RandomForestRegressor(**rf_best_params, random_state=42, n_jobs=-1)
final_model.fit(X_work_scaled, y_work)

final_preds = final_model.predict(X_final_test_scaled)
y_true_test = y_final_test.values

final_r2   = r2_score(y_true_test, final_preds)
final_mae  = mean_absolute_error(y_true_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_true_test, final_preds))
final_mse  = mean_squared_error(y_true_test, final_preds)
final_mape = np.mean(np.abs((y_true_test - final_preds) / y_true_test)) * 100

print(f"R²   : {final_r2:.4f}")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"MSE  : {final_mse:.4f}")
print(f"MAPE : {final_mape:.2f}%")

  0%|          | 0/50 [00:00<?, ?it/s]

Best params : {'n_estimators': 173, 'max_depth': 10, 'min_samples_leaf': 1, 'max_features': 0.48326080093931845}
Best CV R²  : 0.6234
═══════════════════════════════ RANDOM FOREST (Stratified KFold) ═════════════

Fold 1:  R²=0.6794  MAE=0.1748  RMSE=0.2376  MAPE=inf%
Fold 2:  R²=0.6185  MAE=0.1557  RMSE=0.2131  MAPE=75.39%
Fold 3:  R²=0.6049  MAE=0.1857  RMSE=0.2445  MAPE=60.57%
Fold 4:  R²=0.6261  MAE=0.1648  RMSE=0.2227  MAPE=65.58%
Fold 5:  R²=0.5880  MAE=0.2101  RMSE=0.2789  MAPE=134.42%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6234     0.0309     0.5880     0.6794
MAE          0.1782     0.0188     0.1557     0.2101
RMSE         0.2394     0.0226     0.2131     0.2789
MSE          0.0578     0.0112     0.0454     0.0778
MAPE%           inf        nan      60.57        inf

═══════════════════════════════ FINAL TEST SET ════════════════════════════════

R²   : 0.7992
MAE  : 0.1355
RMSE : 0.1722
MSE  : 0

# ══════════════ **XGBOOST** ══════════════

# **STRATIFIED ON XGBOOST**

In [13]:
# Optuna objective with Stratified K-Fold on WORKING data only
def objective_skf(trial, X, y, y_binned):
    model = XGBRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 100, 500),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        max_depth        = trial.suggest_int('max_depth', 3, 8),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda       = trial.suggest_float('reg_lambda', 1, 10),
        random_state=42, verbosity=0,
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y_binned):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        scaler = RobustScaler()
        X_tr[num_cols]  = scaler.fit_transform(X_tr[num_cols])
        X_val[num_cols] = scaler.transform(X_val[num_cols])

        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))

    return np.mean(scores)

# Run Optuna on working data only
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective_skf(trial, X_work, y_work, yb_work),  # ← X_work not X
    n_trials=80,
    show_progress_bar=True
)
xgb_best_params = study.best_params
print("Best params :", xgb_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

# Cross-validation evaluation on working data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_r2, all_mae, all_rmse, all_mse, all_mape = [], [], [], [], []

print("═══════════════════════════════ XGB (Stratified KFold) ═══════════════════════\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_work, yb_work), 1):  # ← X_work
    X_train = X_work.iloc[train_idx].copy()
    X_test  = X_work.iloc[test_idx].copy()
    y_train = y_work.iloc[train_idx]
    y_test  = y_work.iloc[test_idx]

    scaler = RobustScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = XGBRegressor(**xgb_best_params, random_state=42, verbosity=0)
    model.fit(X_train, y_train)
    preds  = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    all_r2.append(r2);    all_mae.append(mae)
    all_rmse.append(rmse); all_mse.append(mse); all_mape.append(mape)

    print(f"Fold {fold}:  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

# CV Summary
print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
for name, vals in [("R²", all_r2), ("MAE", all_mae),
                   ("RMSE", all_rmse), ("MSE", all_mse), ("MAPE%", all_mape)]:
    fmt = ".2f" if name == "MAPE%" else ".4f"
    print(f"{name:<8} {np.mean(vals):>10{fmt}} {np.std(vals):>10{fmt}} "
          f"{np.min(vals):>10{fmt}} {np.max(vals):>10{fmt}}")

# Train final model on ALL working data, evaluate on held-out test
print("\n═══════════════════════════════ FINAL TEST SET ════════════════════════════════\n")

scaler_final = RobustScaler()
X_work_scaled      = X_work.copy()
X_final_test_scaled = X_final_test.copy()

X_work_scaled[num_cols]       = scaler_final.fit_transform(X_work[num_cols])
X_final_test_scaled[num_cols] = scaler_final.transform(X_final_test[num_cols])  # ← no fit!

final_model = XGBRegressor(**xgb_best_params, random_state=42, verbosity=0)
final_model.fit(X_work_scaled, y_work)

final_preds = final_model.predict(X_final_test_scaled)
y_true_test = y_final_test.values

final_r2   = r2_score(y_true_test, final_preds)
final_mae  = mean_absolute_error(y_true_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_true_test, final_preds))
final_mse  = mean_squared_error(y_true_test, final_preds)
final_mape = np.mean(np.abs((y_true_test - final_preds) / y_true_test)) * 100

print(f"R²   : {final_r2:.4f}")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"MSE  : {final_mse:.4f}")
print(f"MAPE : {final_mape:.2f}%")

  0%|          | 0/80 [00:00<?, ?it/s]

Best params : {'n_estimators': 365, 'learning_rate': 0.032452279616908504, 'max_depth': 3, 'subsample': 0.8770775088607893, 'colsample_bytree': 0.6669988444799845, 'reg_lambda': 2.474426977190059}
Best CV R²  : 0.6542
═══════════════════════════════ XGB (Stratified KFold) ═══════════════════════

Fold 1:  R²=0.6816  MAE=0.1713  RMSE=0.2368  MAPE=inf%
Fold 2:  R²=0.6481  MAE=0.1484  RMSE=0.2046  MAPE=76.34%
Fold 3:  R²=0.6425  MAE=0.1871  RMSE=0.2326  MAPE=63.63%
Fold 4:  R²=0.6671  MAE=0.1520  RMSE=0.2102  MAPE=65.83%
Fold 5:  R²=0.6316  MAE=0.1985  RMSE=0.2637  MAPE=134.94%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6542     0.0179     0.6316     0.6816
MAE          0.1714     0.0194     0.1484     0.1985
RMSE         0.2296     0.0211     0.2046     0.2637
MSE          0.0532     0.0099     0.0419     0.0696
MAPE%           inf        nan      63.63        inf

═══════════════════════════════ FINAL TEST SET 

# ══════════════ **LIGHT GBM** ══════════════

# **STRATIFIED ON LIGHT GBM**

In [15]:
# Optuna objective with Stratified K-Fold on WORKING data only
def objective_skf(trial, X, y, y_binned):
    model = LGBMRegressor(
        n_estimators      = trial.suggest_int('n_estimators', 100, 500),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 20, 80),
        max_depth         = trial.suggest_int('max_depth', 3, 10),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 50),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda        = trial.suggest_float('reg_lambda', 1, 10),
        random_state=42, verbose=-1,
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y_binned):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        scaler = RobustScaler()
        X_tr[num_cols]  = scaler.fit_transform(X_tr[num_cols])
        X_val[num_cols] = scaler.transform(X_val[num_cols])

        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))

    return np.mean(scores)

# Run Optuna on working data only
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective_skf(trial, X_work, y_work, yb_work),  # ← X_work not X
    n_trials=80,
    show_progress_bar=True
)
lgbm_best_params = study.best_params
print("Best params :", lgbm_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

# Cross-validation evaluation on working data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_r2, all_mae, all_rmse, all_mse, all_mape = [], [], [], [], []

print("═══════════════════════════════ LGBM (Stratified KFold) ══════════════════════\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_work, yb_work), 1):  # ← X_work
    X_train = X_work.iloc[train_idx].copy()
    X_test  = X_work.iloc[test_idx].copy()
    y_train = y_work.iloc[train_idx]
    y_test  = y_work.iloc[test_idx]

    scaler = RobustScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = LGBMRegressor(**lgbm_best_params, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    preds  = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    all_r2.append(r2);    all_mae.append(mae)
    all_rmse.append(rmse); all_mse.append(mse); all_mape.append(mape)

    print(f"Fold {fold}:  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

# CV Summary
print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
for name, vals in [("R²", all_r2), ("MAE", all_mae),
                   ("RMSE", all_rmse), ("MSE", all_mse), ("MAPE%", all_mape)]:
    fmt = ".2f" if name == "MAPE%" else ".4f"
    print(f"{name:<8} {np.mean(vals):>10{fmt}} {np.std(vals):>10{fmt}} "
          f"{np.min(vals):>10{fmt}} {np.max(vals):>10{fmt}}")

# Train final model on ALL working data, evaluate on held-out test
print("\n═══════════════════════════════ FINAL TEST SET ════════════════════════════════\n")

scaler_final        = RobustScaler()
X_work_scaled       = X_work.copy()
X_final_test_scaled = X_final_test.copy()

X_work_scaled[num_cols]       = scaler_final.fit_transform(X_work[num_cols])
X_final_test_scaled[num_cols] = scaler_final.transform(X_final_test[num_cols])  # ← no fit!

final_model = LGBMRegressor(**lgbm_best_params, random_state=42, verbose=-1)
final_model.fit(X_work_scaled, y_work)

final_preds = final_model.predict(X_final_test_scaled)
y_true_test = y_final_test.values

final_r2   = r2_score(y_true_test, final_preds)
final_mae  = mean_absolute_error(y_true_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_true_test, final_preds))
final_mse  = mean_squared_error(y_true_test, final_preds)
final_mape = np.mean(np.abs((y_true_test - final_preds) / y_true_test)) * 100

print(f"R²   : {final_r2:.4f}")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"MSE  : {final_mse:.4f}")
print(f"MAPE : {final_mape:.2f}%")

  0%|          | 0/80 [00:00<?, ?it/s]

Best params : {'n_estimators': 474, 'learning_rate': 0.016784084202113768, 'num_leaves': 50, 'max_depth': 3, 'min_child_samples': 7, 'subsample': 0.513871390810811, 'colsample_bytree': 0.8788070830576229, 'reg_lambda': 2.9647280489835683}
Best CV R²  : 0.6460
═══════════════════════════════ LGBM (Stratified KFold) ══════════════════════

Fold 1:  R²=0.6763  MAE=0.1687  RMSE=0.2388  MAPE=inf%
Fold 2:  R²=0.6609  MAE=0.1500  RMSE=0.2009  MAPE=76.57%
Fold 3:  R²=0.6176  MAE=0.1858  RMSE=0.2406  MAPE=62.34%
Fold 4:  R²=0.6526  MAE=0.1576  RMSE=0.2147  MAPE=65.63%
Fold 5:  R²=0.6224  MAE=0.1968  RMSE=0.2670  MAPE=136.55%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6460     0.0226     0.6176     0.6763
MAE          0.1718     0.0174     0.1500     0.1968
RMSE         0.2324     0.0229     0.2009     0.2670
MSE          0.0545     0.0107     0.0403     0.0713
MAPE%           inf        nan      62.34        inf

═════

# ══════════════ **SVR** ══════════════

# **STRATIFIED ON SVR**

In [16]:
# Optuna objective with Stratified K-Fold on WORKING data only
def objective_skf(trial, X, y, y_binned):
    model = SVR(
        C       = trial.suggest_float('C', 0.1, 100, log=True),
        epsilon = trial.suggest_float('epsilon', 0.01, 1.0, log=True),
        gamma   = trial.suggest_categorical('gamma', ['scale', 'auto']),
        kernel  = trial.suggest_categorical('kernel', ['rbf', 'linear', 'poly']),
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y_binned):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        scaler = RobustScaler()
        X_tr[num_cols]  = scaler.fit_transform(X_tr[num_cols])
        X_val[num_cols] = scaler.transform(X_val[num_cols])

        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))

    return np.mean(scores)

# Run Optuna on working data only
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective_skf(trial, X_work, y_work, yb_work),  # ← X_work not X
    n_trials=80,
    show_progress_bar=True
)
svr_best_params = study.best_params
print("Best params :", svr_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

# Cross-validation evaluation on working data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_r2, all_mae, all_rmse, all_mse, all_mape = [], [], [], [], []

print("═══════════════════════════════ SVR (Stratified KFold) ═══════════════════════\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_work, yb_work), 1):  # ← X_work
    X_train = X_work.iloc[train_idx].copy()
    X_test  = X_work.iloc[test_idx].copy()
    y_train = y_work.iloc[train_idx]
    y_test  = y_work.iloc[test_idx]

    scaler = RobustScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = SVR(**svr_best_params)
    model.fit(X_train, y_train)
    preds  = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    all_r2.append(r2);    all_mae.append(mae)
    all_rmse.append(rmse); all_mse.append(mse); all_mape.append(mape)

    print(f"Fold {fold}:  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

# CV Summary
print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
for name, vals in [("R²", all_r2), ("MAE", all_mae),
                   ("RMSE", all_rmse), ("MSE", all_mse), ("MAPE%", all_mape)]:
    fmt = ".2f" if name == "MAPE%" else ".4f"
    print(f"{name:<8} {np.mean(vals):>10{fmt}} {np.std(vals):>10{fmt}} "
          f"{np.min(vals):>10{fmt}} {np.max(vals):>10{fmt}}")

# Train final model on ALL working data, evaluate on held-out test
print("\n═══════════════════════════════ FINAL TEST SET ════════════════════════════════\n")

scaler_final        = RobustScaler()
X_work_scaled       = X_work.copy()
X_final_test_scaled = X_final_test.copy()

X_work_scaled[num_cols]       = scaler_final.fit_transform(X_work[num_cols])
X_final_test_scaled[num_cols] = scaler_final.transform(X_final_test[num_cols])  # ← no fit!

final_model = SVR(**svr_best_params)
final_model.fit(X_work_scaled, y_work)

final_preds = final_model.predict(X_final_test_scaled)
y_true_test = y_final_test.values

final_r2   = r2_score(y_true_test, final_preds)
final_mae  = mean_absolute_error(y_true_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_true_test, final_preds))
final_mse  = mean_squared_error(y_true_test, final_preds)
final_mape = np.mean(np.abs((y_true_test - final_preds) / y_true_test)) * 100

print(f"R²   : {final_r2:.4f}")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"MSE  : {final_mse:.4f}")
print(f"MAPE : {final_mape:.2f}%")

  0%|          | 0/80 [00:00<?, ?it/s]

Best params : {'C': 3.998397263200587, 'epsilon': 0.05148956595019504, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV R²  : 0.6276
═══════════════════════════════ SVR (Stratified KFold) ═══════════════════════

Fold 1:  R²=0.6554  MAE=0.1792  RMSE=0.2463  MAPE=inf%
Fold 2:  R²=0.6166  MAE=0.1474  RMSE=0.2136  MAPE=76.60%
Fold 3:  R²=0.6847  MAE=0.1698  RMSE=0.2185  MAPE=63.53%
Fold 4:  R²=0.5946  MAE=0.1555  RMSE=0.2319  MAPE=65.69%
Fold 5:  R²=0.5867  MAE=0.2066  RMSE=0.2793  MAPE=137.73%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6276     0.0372     0.5867     0.6847
MAE          0.1717     0.0206     0.1474     0.2066
RMSE         0.2379     0.0236     0.2136     0.2793
MSE          0.0572     0.0117     0.0456     0.0780
MAPE%           inf        nan      63.53        inf

═══════════════════════════════ FINAL TEST SET ════════════════════════════════

R²   : 0.8047
MAE  : 0.1343
RMSE : 0.1698
MSE  : 0.0288
M

# ══════════════ **GPR** ══════════════

# **STRATIFIED ON GPR**

In [21]:
# Define kernel
GPR_KERNEL = (
    C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5)
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
)

# Cross-validation evaluation on working data
# There are no hyperparametrs in GPR to tune via Optuna (kernel handles everything internally via log-marginal-likelihood optimization), so we skip Optuna and go to cross validation

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_r2, all_mae, all_rmse, all_mse, all_mape = [], [], [], [], []

print("═══════════════════════════════ GPR (Stratified KFold) ════════════════════════\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_work, yb_work), 1):  # ← X_work
    X_train = X_work.iloc[train_idx].copy()
    X_test  = X_work.iloc[test_idx].copy()
    y_train = y_work.iloc[train_idx]
    y_test  = y_work.iloc[test_idx]

    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])  # ← no fit!

    # Fresh kernel per fold to avoid warm-starting from previous fold
    kernel = (
        C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5)
        + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
    )

    model = GaussianProcessRegressor(
        kernel               = kernel,
        alpha                = 1e-6,
        normalize_y          = True,
        n_restarts_optimizer = 5,
        random_state         = 42
    )

    model.fit(X_train, y_train)
    preds  = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    all_r2.append(r2);    all_mae.append(mae)
    all_rmse.append(rmse); all_mse.append(mse); all_mape.append(mape)

    print(f"Fold {fold}:  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

# CV Summary
print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
for name, vals in [("R²", all_r2), ("MAE", all_mae),
                   ("RMSE", all_rmse), ("MSE", all_mse), ("MAPE%", all_mape)]:
    fmt = ".2f" if name == "MAPE%" else ".4f"
    print(f"{name:<8} {np.mean(vals):>10{fmt}} {np.std(vals):>10{fmt}} "
          f"{np.min(vals):>10{fmt}} {np.max(vals):>10{fmt}}")

# Train final model on ALL working data, evaluate on held-out test
print("\n═══════════════════════════════ FINAL TEST SET ════════════════════════════════\n")

scaler_final        = StandardScaler()
X_work_scaled       = X_work.copy()
X_final_test_scaled = X_final_test.copy()

X_work_scaled[num_cols]       = scaler_final.fit_transform(X_work[num_cols])
X_final_test_scaled[num_cols] = scaler_final.transform(X_final_test[num_cols])  # ← no fit!

final_model = GaussianProcessRegressor(
    kernel               = GPR_KERNEL,
    alpha                = 1e-6,
    normalize_y          = True,
    n_restarts_optimizer = 5,
    random_state         = 42
)
final_model.fit(X_work_scaled, y_work)

final_preds = final_model.predict(X_final_test_scaled)
y_true_test = y_final_test.values

final_r2   = r2_score(y_true_test, final_preds)
final_mae  = mean_absolute_error(y_true_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_true_test, final_preds))
final_mse  = mean_squared_error(y_true_test, final_preds)
final_mape = np.mean(np.abs((y_true_test - final_preds) / y_true_test)) * 100

print(f"R²   : {final_r2:.4f}")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"MSE  : {final_mse:.4f}")
print(f"MAPE : {final_mape:.2f}%")

═══════════════════════════════ GPR (Stratified KFold) ════════════════════════

Fold 1:  R²=0.6493  MAE=0.1789  RMSE=0.2485  MAPE=inf%
Fold 2:  R²=0.6223  MAE=0.1487  RMSE=0.2120  MAPE=77.05%
Fold 3:  R²=0.6732  MAE=0.1703  RMSE=0.2224  MAPE=63.67%
Fold 4:  R²=0.6016  MAE=0.1590  RMSE=0.2299  MAPE=66.42%
Fold 5:  R²=0.5555  MAE=0.2213  RMSE=0.2897  MAPE=138.13%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6204     0.0405     0.5555     0.6732
MAE          0.1757     0.0250     0.1487     0.2213
RMSE         0.2405     0.0273     0.2120     0.2897
MSE          0.0586     0.0138     0.0449     0.0839
MAPE%           inf        nan      63.67        inf

═══════════════════════════════ FINAL TEST SET ════════════════════════════════

R²   : 0.7935
MAE  : 0.1361
RMSE : 0.1747
MSE  : 0.0305
MAPE : inf%


# ══════════════ **STACKING** ══════════════

In [24]:

# Define GPR kernel
GPR_KERNEL = (
    ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5)
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
)

# Cross-validation evaluation on working data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_r2, all_mae, all_rmse, all_mse, all_mape = [], [], [], [], []

print("═══════════════════════════════ Stacking (Stratified KFold) ══════════════════\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_work, yb_work), 1):
    X_train = X_work.iloc[train_idx].copy()
    X_test  = X_work.iloc[test_idx].copy()
    y_train = y_work.iloc[train_idx]
    y_test  = y_work.iloc[test_idx]

    scaler = RobustScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])  # ← no fit!

    # Fresh kernel every fold to avoid warm-starting leakage
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5)
        + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
    )

    stack = StackingRegressor(
        estimators=[
            #('rf',   RandomForestRegressor(**rf_best_params,   random_state=42, n_jobs=-1)),
            ('lgbm', LGBMRegressor(**lgbm_best_params,         random_state=42, verbose=-1, n_jobs=-1)),
            ('xgb',  XGBRegressor(**xgb_best_params,           random_state=42, verbosity=0, n_jobs=-1)),
            #('svr',  SVR(**svr_best_params)),
            ('gpr',  GaussianProcessRegressor(
                         kernel               = kernel,
                         alpha                = 1e-6,
                         normalize_y          = True,
                         n_restarts_optimizer = 3,
                         random_state         = 42)),
        ],
        final_estimator = Ridge(alpha=1.0),
        cv              = 5,
        n_jobs          = -1
    )

    stack.fit(X_train, y_train)
    preds  = stack.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    all_r2.append(r2);    all_mae.append(mae)
    all_rmse.append(rmse); all_mse.append(mse); all_mape.append(mape)

    print(f"Fold {fold}:  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

# CV Summary
print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
for name, vals in [("R²", all_r2), ("MAE", all_mae),
                   ("RMSE", all_rmse), ("MSE", all_mse), ("MAPE%", all_mape)]:
    fmt = ".2f" if name == "MAPE%" else ".4f"
    print(f"{name:<8} {np.mean(vals):>10{fmt}} {np.std(vals):>10{fmt}} "
          f"{np.min(vals):>10{fmt}} {np.max(vals):>10{fmt}}")

# Train final model on ALL working data, evaluate on held-out test
print("\n═══════════════════════════════ FINAL TEST SET ════════════════════════════════\n")

scaler_final        = RobustScaler()
X_work_scaled       = X_work.copy()
X_final_test_scaled = X_final_test.copy()

X_work_scaled[num_cols]       = scaler_final.fit_transform(X_work[num_cols])
X_final_test_scaled[num_cols] = scaler_final.transform(X_final_test[num_cols])  # ← no fit!

final_stack = StackingRegressor(
    estimators=[
        ('rf',   RandomForestRegressor(**rf_best_params,   random_state=42, n_jobs=-1)),
        ('lgbm', LGBMRegressor(**lgbm_best_params,         random_state=42, verbose=-1, n_jobs=-1)),
        ('xgb',  XGBRegressor(**xgb_best_params,           random_state=42, verbosity=0, n_jobs=-1)),
        ('svr',  SVR(**svr_best_params)),
        ('gpr',  GaussianProcessRegressor(
                     kernel               = GPR_KERNEL,
                     alpha                = 1e-6,
                     normalize_y          = True,
                     n_restarts_optimizer = 3,
                     random_state         = 42)),
    ],
    final_estimator = Ridge(alpha=1.0),
    cv              = 5,
    n_jobs          = -1
)

final_stack.fit(X_work_scaled, y_work)

final_preds = final_stack.predict(X_final_test_scaled)
y_true_test = y_final_test.values

final_r2   = r2_score(y_true_test, final_preds)
final_mae  = mean_absolute_error(y_true_test, final_preds)
final_rmse = np.sqrt(mean_squared_error(y_true_test, final_preds))
final_mse  = mean_squared_error(y_true_test, final_preds)
final_mape = np.mean(np.abs((y_true_test - final_preds) / y_true_test)) * 100

print(f"R²   : {final_r2:.4f}")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"MSE  : {final_mse:.4f}")
print(f"MAPE : {final_mape:.2f}%")

═══════════════════════════════ Stacking (Stratified KFold) ══════════════════

Fold 1:  R²=0.6883  MAE=0.1680  RMSE=0.2343  MAPE=inf%
Fold 2:  R²=0.6660  MAE=0.1433  RMSE=0.1993  MAPE=76.81%
Fold 3:  R²=0.6632  MAE=0.1783  RMSE=0.2258  MAPE=63.55%
Fold 4:  R²=0.6679  MAE=0.1474  RMSE=0.2099  MAPE=66.10%
Fold 5:  R²=0.6191  MAE=0.2021  RMSE=0.2682  MAPE=137.84%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6609     0.0227     0.6191     0.6883
MAE          0.1678     0.0215     0.1433     0.2021
RMSE         0.2275     0.0237     0.1993     0.2682
MSE          0.0523     0.0111     0.0397     0.0719
MAPE%           inf        nan      63.55        inf

═══════════════════════════════ FINAL TEST SET ════════════════════════════════

R²   : 0.8089
MAE  : 0.1339
RMSE : 0.1680
MSE  : 0.0282
MAPE : inf%
